# AI Meeting Notes Analyzer

This notebook implements a LangGraph workflow for extracting topics, a concise summary, assigned actions, and urgency from a meeting transcript. Its schemas and validation logic use plain dictionaries and functions rather than custom classes. The chat model is accessed through OpenRouter; transcript embeddings are generated locally with Sentence Transformers.

Set `OPENROUTER_API_KEY` in your environment before running the analysis cell. The Streamlit app loads the workflow definition from the tagged runtime cell below.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from functools import lru_cache
from pathlib import Path
from typing import Any

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from dotenv import load_dotenv

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
load_dotenv(Path.cwd() / ".env")


PRIORITY_LEVELS = ["High Priority", "Medium Priority", "Low Priority"]

OUTPUT_SCHEMAS = {
    "topics": {
        "type": "object",
        "properties": {"topics": {"type": "array", "items": {"type": "string"}}},
        "required": ["topics"],
        "additionalProperties": False,
    },
    "summary": {
        "type": "object",
        "properties": {"summary": {"type": "string"}},
        "required": ["summary"],
        "additionalProperties": False,
    },
    "actions": {
        "type": "object",
        "properties": {
            "action_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task_description": {"type": "string"},
                        "owner": {"type": "string"},
                    },
                    "required": ["task_description"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["action_items"],
        "additionalProperties": False,
    },
    "priorities": {
        "type": "object",
        "properties": {
            "overall_priority": {"type": "string", "enum": PRIORITY_LEVELS},
            "task_priorities": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task_description": {"type": "string"},
                        "priority": {"type": "string", "enum": PRIORITY_LEVELS},
                    },
                    "required": ["task_description", "priority"],
                    "additionalProperties": False,
                },
            },
        },
        "required": ["overall_priority", "task_priorities"],
        "additionalProperties": False,
    },
}


def validate_json_value(value: Any, schema: dict[str, Any], path: str = "result") -> None:
    """Validate decoded JSON against the subset of JSON Schema used here."""
    expected_type = schema["type"]
    type_checks = {
        "object": lambda item: isinstance(item, dict),
        "array": lambda item: isinstance(item, list),
        "string": lambda item: isinstance(item, str),
    }
    if expected_type not in type_checks or not type_checks[expected_type](value):
        raise ValueError(f"{path} must be a JSON {expected_type}.")

    if "enum" in schema and value not in schema["enum"]:
        raise ValueError(f"{path} must be one of {schema['enum']}.")

    if expected_type == "object":
        properties = schema.get("properties", {})
        missing = [key for key in schema.get("required", []) if key not in value]
        if missing:
            raise ValueError(f"{path} is missing required fields: {', '.join(missing)}.")
        if schema.get("additionalProperties") is False:
            extra = set(value) - set(properties)
            if extra:
                raise ValueError(f"{path} contains unexpected fields: {', '.join(sorted(extra))}.")
        for key, item in value.items():
            if key in properties:
                validate_json_value(item, properties[key], f"{path}.{key}")
    elif expected_type == "array":
        item_schema = schema.get("items")
        if item_schema:
            for index, item in enumerate(value):
                validate_json_value(item, item_schema, f"{path}[{index}]")


def normalize_owner(owner: Any) -> str:
    """Normalize absent or common unknown-owner values consistently."""
    if not isinstance(owner, str) or owner.strip().casefold() in {
        "", "not specified", "owner not specified", "unknown", "n/a", "none"
    }:
        return "Owner: Not specified"
    return owner.strip()


def validate_summary(summary: str) -> str:
    """Require a non-empty summary containing three to five sentences."""
    summary = summary.strip()
    sentence_count = len([part for part in re.split(r"(?<=[.!?])\s+", summary) if part])
    if not 3 <= sentence_count <= 5:
        raise ValueError("The meeting summary must contain 3 to 5 sentences.")
    return summary


@lru_cache(maxsize=2)
def get_local_embeddings(model_name: str = DEFAULT_EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Load an embedding model that runs locally on the CPU."""
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def split_transcript(transcript: str) -> list[str]:
    segments = [part.strip() for part in re.split(r"(?:\r?\n)+|(?<=[.!?])\s+", transcript) if part.strip()]
    return segments or [transcript.strip()]


def build_workflow(
    api_key: str,
    model_name: str = DEFAULT_MODEL,
    embedding_model_name: str = DEFAULT_EMBEDDING_MODEL,
):
    """Build and compile the meeting-analysis LangGraph."""
    if not api_key.strip():
        raise ValueError("An OpenRouter API key is required. Set OPENROUTER_API_KEY or enter it in the app.")

    chat_model = ChatOpenAI(
        model=model_name,
        api_key=api_key,
        base_url=OPENROUTER_BASE_URL,
        temperature=0,
        timeout=90,
    )
    embeddings = get_local_embeddings(embedding_model_name)

    def invoke_structured(schema_name: str, instructions: str) -> dict[str, Any]:
        schema = OUTPUT_SCHEMAS[schema_name]
        prompt = (
            f"{instructions}\n\n"
            "Return only a valid JSON object matching this schema exactly:\n"
            f"{json.dumps(schema, ensure_ascii=False)}"
        )
        result = chat_model.with_structured_output(None, method="json_mode").invoke(prompt)
        validate_json_value(result, schema, schema_name)
        return result

    def evidence_for(
        query: str, segments: list[str], segment_vectors: list[list[float]], limit: int = 8
    ) -> str:
        query_vector = embeddings.embed_query(query)
        ranked = []
        for segment, vector in zip(segments, segment_vectors):
            denominator = math.sqrt(sum(value * value for value in query_vector)) * math.sqrt(
                sum(value * value for value in vector)
            )
            similarity = sum(left * right for left, right in zip(query_vector, vector)) / denominator if denominator else 0
            ranked.append((similarity, segment))
        selected = [segment for _, segment in sorted(ranked, reverse=True)[:limit]]
        return "\n".join(f"- {segment}" for segment in selected)

    def input_node(state: dict[str, Any]) -> dict[str, Any]:
        transcript = state.get("transcript", "").strip()
        if not transcript:
            raise ValueError("The meeting transcript cannot be empty.")
        segments = split_transcript(transcript)
        embedding_vectors = embeddings.embed_documents(segments)
        return {
            **state,
            "transcript": transcript,
            "segments": segments,
            "embedding_vectors": embedding_vectors,
        }

    def topics_node(state: dict[str, Any]) -> dict[str, Any]:
        result = invoke_structured(
            "topics",
            "Extract the distinct main discussion topics from this meeting. Return short, specific topic labels. "
            "Do not invent topics.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('main discussion topics and themes', state['segments'], state['embedding_vectors'])}"
        )
        result["topics"] = [topic.strip() for topic in result["topics"] if topic.strip()]
        return {**state, **result}

    def summary_node(state: dict[str, Any]) -> dict[str, Any]:
        result = invoke_structured(
            "summary",
            "Summarize what happened in this meeting in 3 to 5 concise sentences. Preserve important decisions "
            "and context; do not add unsupported facts.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('meeting purpose, outcomes, and decisions', state['segments'], state['embedding_vectors'])}"
        )
        result["summary"] = validate_summary(result["summary"])
        return {**state, **result}

    def actions_node(state: dict[str, Any]) -> dict[str, Any]:
        result = invoke_structured(
            "actions",
            "Extract only concrete tasks that someone explicitly commits to or is assigned during the meeting. "
            "Do not treat suggestions, questions, or unassigned possibilities as tasks. For an unknown owner, use "
            "exactly 'Owner: Not specified'. Preserve each task as a concise action.\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('assigned tasks, commitments, and responsible people', state['segments'], state['embedding_vectors'])}"
        )
        result["action_items"] = [
            {
                "task_description": item["task_description"].strip(),
                "owner": normalize_owner(item.get("owner")),
            }
            for item in result["action_items"]
        ]
        if any(not item["task_description"] for item in result["action_items"]):
            raise ValueError("Action-item task descriptions cannot be empty.")
        return {**state, **result}

    def priority_node(state: dict[str, Any]) -> dict[str, Any]:
        result = invoke_structured(
            "priorities",
            "Classify urgency for every action item. Use High Priority for explicit urgent/ASAP language, same-day "
            "deadlines, critical customer or service impact, or deadlines by tomorrow. Use Medium Priority for "
            "near-term deadlines such as this week or Friday, or clear but non-urgent commitments. Use Low Priority "
            "when no urgency or deadline is stated. Return one priority per supplied task and an overall priority "
            "reflecting the most urgent task. Do not invent tasks.\n\n"
            f"Action items:\n{[item['task_description'] + ' — ' + item['owner'] for item in state['action_items']]}\n\n"
            f"Transcript:\n{state['transcript']}\n\n"
            f"Locally retrieved transcript excerpts:\n{evidence_for('task urgency deadlines and priority', state['segments'], state['embedding_vectors'])}"
        )
        if len(result["task_priorities"]) != len(state["action_items"]):
            raise ValueError("The priority agent must return exactly one priority for every action item.")
        for action, priority in zip(state["action_items"], result["task_priorities"], strict=True):
            priority["task_description"] = action["task_description"]
        return {
            **state,
            "overall_priority": result["overall_priority"],
            "task_priorities": result["task_priorities"],
        }

    def route_after_actions(state: dict[str, Any]) -> str:
        return "priority" if state.get("action_items") else "finalize"

    def finalize_node(state: dict[str, Any]) -> dict[str, Any]:
        priorities = state.get("task_priorities", [])
        actions = [
            {**action, "priority": priority["priority"]}
            for action, priority in zip(state.get("action_items", []), priorities, strict=True)
        ]
        overall_priority = state.get("overall_priority") or "No action items identified in this meeting."
        return {
            **state,
            "report": {
                "meeting_summary": state["summary"],
                "key_topics": state["topics"],
                "action_items": actions,
                "task_owners": [
                    {"task_description": item["task_description"], "owner": item["owner"]}
                    for item in actions
                ],
                "priority_level": overall_priority,
            }
        }

    graph = StateGraph(dict)
    graph.add_node("input", input_node)
    graph.add_node("topics", topics_node)
    graph.add_node("summary", summary_node)
    graph.add_node("actions", actions_node)
    graph.add_node("priority", priority_node)
    graph.add_node("finalize", finalize_node)
    graph.add_edge(START, "input")
    graph.add_edge("input", "topics")
    graph.add_edge("topics", "summary")
    graph.add_edge("summary", "actions")
    graph.add_conditional_edges("actions", route_after_actions, {"priority": "priority", "finalize": "finalize"})
    graph.add_edge("priority", "finalize")
    graph.add_edge("finalize", END)
    return graph.compile()


## Analyze a transcript

Run the following cell after setting `OPENROUTER_API_KEY`. The first analysis downloads the small embedding model once; embeddings and similarity retrieval run locally.

In [ ]:
from pathlib import Path
import os

transcript_path = Path("Meeting_Transcript.txt")
transcript = transcript_path.read_text(encoding="utf-8")
analyzer = build_workflow(api_key=os.environ["OPENROUTER_API_KEY"])
result = analyzer.invoke({"transcript": transcript})
result["report"]